In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Read Data from Silver Layer**

In [0]:
silver_df = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

In [0]:
display(silver_df)

### **Selecting Required Columns**

In [0]:
gold_df = silver_df.select("customer_id", "customer_name","city", "customer", "market", "platform", "channel")

In [0]:
display(gold_df)

### **Write the Final Data Frame to Gold Schema**

In [0]:
gold_df.write \
  .format("delta") \
    .option("Delta.enableChangeDataFeed", True) \
      .mode("overwrite") \
        .saveAsTable(f"{catalog}.{gold_schema}.sb_dim{data_source}")

### **Merging Child Company Data with Parent Company**

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customer")
df_child_customers = spark.read.table("fmcg.gold.sb_dimcustomers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")

delta_table.alias("target") \
    .merge(df_child_customers.alias("source"), \
        "target.customer_code = source.customer_code") \
            .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                    .execute()